In [ ]:
import streamlit as st
import requests
import os
from transformers import pipeline

SERP_API_KEY = os.getenv("SERP_API_KEY") or "e836aa46bfc5ac99d4d3f4250cc0e0b3669f21ceef6a66422a83ce8c181f648e"

@st.cache_resource
def load_model():
    return pipeline("zero-shot-classification")

classifier = load_model()

def search_news(query):
    url = "https://serpapi.com/search.json"
    params = {
        "q": query,
        "api_key": SERP_API_KEY,
        "num": 3
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        st.error("❌ Error fetching data from API")
        return []

    results = []
    if "organic_results" in data:
        for item in data["organic_results"][:3]:
            results.append({
                "title": item.get("title", "No title"),
                "snippet": item.get("snippet", "No description"),
                "link": item.get("link", "#")
            })
    return results


def fact_check(news_text, articles):
    context = " ".join([a["snippet"] for a in articles])

    result = classifier(
        f"Claim: {news_text}\nEvidence: {context}",
        candidate_labels=["fake news", "real news"]
    )

    return result


st.set_page_config(
    page_title="AI Fake News Detector",
    page_icon="🧠",
    layout="centered"
)

st.markdown("""
<style>
.stApp {
    background-color: #0E1117;
}
textarea {
    background-color: #262730 !important;
    color: white !important;
}
.stButton button {
    width: 100%;
    border-radius: 10px;
    background-color: #4CAF50;
    color: white;
    font-weight: bold;
}
</style>
""", unsafe_allow_html=True)

st.title("🧠 AI Fake News Detector")
st.caption("RAG + Real-Time Fact Checking System")

st.markdown("---")


user_input = st.text_area("📝 Enter News Claim", height=150)

if st.button("🔍 Verify News"):
    if not user_input.strip():
        st.warning("⚠ Please enter some news text")

    else:
        with st.spinner("🔎 Fetching real-time data..."):
            articles = search_news(user_input)

        st.markdown("## 🌐 Retrieved Evidence")

        if not articles:
            st.error("❌ No evidence found")
        else:
            for a in articles:
                st.markdown(f"""
                **📰 {a['title']}**
                {a['snippet']}
                🔗 [Read more]({a['link']})
                """)
                st.markdown("---")

            with st.spinner("🧠 Analyzing..."):
                result = fact_check(user_input, articles)

            label = result["labels"][0]
            score = float(result["scores"][0])

            st.markdown("## 🧾 Final Verdict")

            if label == "fake news":
                st.error("🚨 FAKE NEWS")
            else:
                st.success("✅ REAL NEWS")

            st.markdown("### 📊 Confidence Score")
            st.progress(score)
            st.write(f"Confidence: **{score:.2f}**")

            st.markdown("## 💡 Explanation")
            st.info("Decision is based on comparison between your claim and real-world retrieved evidence.")